In [3]:
import _referAsMain

import math
import time
import random
import sys
import numpy
from IPython.display import SVG, display
from holo.pointers import Pointer
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint, prettyTime

print(sys.version_info)

import torch
from torch.utils.data import DataLoader

from paths_cfg import TOKENIZER_SAVE_DIRECTORY, OUR_DATASET_DIRECTORY
from dataset import svg_dataset
import tokenizer_pfe.tokenizer_project as tokenizerLib
import metrics.metrics

from LLM.model import Model, GenerationStats
import LLM.nanochat.gpt as nanoChatModel
from LLM.nanochat.common import compute_init, autodetect_device_type

added '/autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation' to import paths
sys.version_info(major=3, minor=10, micro=19, releaselevel='final', serial=0)


In [4]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Device count: 1


In [5]:
import importlib
import LLM.model

_ = importlib.reload(LLM.model)
from LLM.model import Model

In [8]:
try:
    torch.cuda.empty_cache()
    del model  # type: ignore
    torch.cuda.empty_cache()
except Exception:
    pass
model = Model.load("model_20.5M", versionID=1, device=torch.device("cuda:0"))
model.show_infos()

loading the tokenizer from: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation/models_save/model_20.5M/tokenizer.json
loading the historique from: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation/models_save/model_20.5M/versions/version_0001_trained-1epoches/history.json
Padding vocab_size from 585 to 640 for efficiency
Scaling the LR for the AdamW parameters ∝1/√(512/768) = 1.224745
GPTConfig(sequence_len=8192, vocab_size=585, n_layer=6, n_head=2, n_kv_head=2, n_embd=512, window_pattern='SSSL')
20_512_972 total params (embeding: 1_310_720 | last layer: 327_680 | transformer: 18_874_560)
on device: device(type='cuda', index=0), with effective context_size: 4096


In [9]:
print(f"trained for {model.nb_epoches_done} epoches")
for k, v in model.historique.get_all_historique().items():
    print(f"{k}: {v.get(str(model.nb_epoches_done-1), None):.4g}")  # type: ignore

trained for 1 epoches
CE_train: 1
CE2_train: 0.7679
PPL_train: 2.72
PPL2_train: 2.155
BPC_train: 0.7312
ENTROPY_train: 1.003
LOGITS_SD_train: 2.899
TOP-1_train: 0.7553
TOP-5_train: 0.829
epoch_duration: 779.3


In [10]:
try:
    if "dataset" in globals():
        raise FileExistsError
    dataset = svg_dataset.SVGDataset(
        OUR_DATASET_DIRECTORY,
        context_size=model.context_size,
        tokenizer=model.tokenizer.encode,
        decoder=model.tokenizer.decode,
    )
except FileExistsError:
    pass

In [12]:
N = 45
print(f"using: {dataset.samples[N].svg_file}")
start = dataset.samples[N].txt[: model.context_size]
# print(start); break
statsPtr: Pointer[GenerationStats] = Pointer()
results = []
for txt in model.generate_flow(
    start=start,
    decode_batch=256,
    temperature=1.0,
    top_k=5,
    max_tokens=320000,
    max_time=2 * 60,
    statsPtr=statsPtr,
):
    results.append(txt)
    print(txt, end="", flush=True)

using: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation/dataset/samples/0045_circle_packing.svg
5.0316)" style="fill:none;" rx="16.5367" cx="0" ry="16.0753" cy="0"/></g><g style="stroke-linecap:round;"><circle r="11.0746" style="fill:none;" cx="718.8503" cy="789.4907"/><circle r="0.8398" style="fill:none;" cx="718.8503" cy="789.4907"/><ellipse rx="38.0399" ry="35.8539" cy="377.87"/><ellipse rx="35.0793" style="fill:none;" cx="718.8503" cy="377.8739"/><ellipse rx="30.5385" ry="30.8539" style="fill:none;" cx="377.8739" cy="330.3015"/><ellipse rx="27.9904" ry="24.9904" style="fill:none;" cx="330.3015" cy="330.3016"/><ellipse transform="translate(0.07"/><ellipse rx="13.0707" ry="24.8799" style="fill:none;" cx="330.3011" cy="330.3016"/><ellipse transform="translate(0" style="fill:none;" cx="330.3011" cy="330.3015"/><ellipse rx="13.0753" style="fill:none;" cx="331.3011" cy="789.6785"/><ellipse rx="8.0377" style="fill:none;" cx="331.3011" cy="789.6785"/><ellipse transfo

In [20]:
print(start)

<svg xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" style="fill-opacity:1; color-rendering:auto; color-interpolation:auto; text-rendering:auto; stroke:black; stroke-linecap:square; stroke-miterlimit:10; shape-rendering:auto; stroke-opacity:1; fill:black; stroke-dasharray:none; font-weight:normal; stroke-width:1; font-family:'Dialog'; font-style:normal; stroke-linejoin:miter; font-size:12px; stroke-dashoffset:0; image-rendering:auto;" width="900" height="900"><defs id="genericDefs"/><g><g style="stroke-linecap:round; fill:white; stroke:white;"><rect x="0" width="900" height="900" y="0" style="stroke:none;"/></g><g style="stroke-linecap:round;" transform="translate(431.0456,173.9697) rotate(350.1284)"><ellipse rx="74.4221" ry="48.0768" style="fill:none;" cx="0" cy="0"/><ellipse rx="70.2875" ry="45.4058" style="fill:none;" cx="0" cy="0"/><ellipse rx="66.153" ry="42.7349" style="fill:none;" cx="0" cy="0"/><ellipse rx="62.0184" ry="40.064" style="fill:none;" c

In [14]:
results.insert(0, start)

In [15]:
print(statsPtr.value)
print(f" -> {statsPtr.value.nb_tokens / statsPtr.value.gen_time:.2f} tokens/sec")

GenerationStats(nb_tokens=6947, gen_time=120.00978818399926, stop_reason='reached max_time')
 -> 57.89 tokens/sec


In [16]:
print(results)

['<svg xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" style="fill-opacity:1; color-rendering:auto; color-interpolation:auto; text-rendering:auto; stroke:black; stroke-linecap:square; stroke-miterlimit:10; shape-rendering:auto; stroke-opacity:1; fill:black; stroke-dasharray:none; font-weight:normal; stroke-width:1; font-family:\'Dialog\'; font-style:normal; stroke-linejoin:miter; font-size:12px; stroke-dashoffset:0; image-rendering:auto;" width="900" height="900"><defs id="genericDefs"/><g><g style="stroke-linecap:round; fill:white; stroke:white;"><rect x="0" width="900" height="900" y="0" style="stroke:none;"/></g><g style="stroke-linecap:round;" transform="translate(431.0456,173.9697) rotate(350.1284)"><ellipse rx="74.4221" ry="48.0768" style="fill:none;" cx="0" cy="0"/><ellipse rx="70.2875" ry="45.4058" style="fill:none;" cx="0" cy="0"/><ellipse rx="66.153" ry="42.7349" style="fill:none;" cx="0" cy="0"/><ellipse rx="62.0184" ry="40.064" style="fill:none

In [17]:
print(results[0])

<svg xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" style="fill-opacity:1; color-rendering:auto; color-interpolation:auto; text-rendering:auto; stroke:black; stroke-linecap:square; stroke-miterlimit:10; shape-rendering:auto; stroke-opacity:1; fill:black; stroke-dasharray:none; font-weight:normal; stroke-width:1; font-family:'Dialog'; font-style:normal; stroke-linejoin:miter; font-size:12px; stroke-dashoffset:0; image-rendering:auto;" width="900" height="900"><defs id="genericDefs"/><g><g style="stroke-linecap:round; fill:white; stroke:white;"><rect x="0" width="900" height="900" y="0" style="stroke:none;"/></g><g style="stroke-linecap:round;" transform="translate(431.0456,173.9697) rotate(350.1284)"><ellipse rx="74.4221" ry="48.0768" style="fill:none;" cx="0" cy="0"/><ellipse rx="70.2875" ry="45.4058" style="fill:none;" cx="0" cy="0"/><ellipse rx="66.153" ry="42.7349" style="fill:none;" cx="0" cy="0"/><ellipse rx="62.0184" ry="40.064" style="fill:none;" c

In [18]:
print("".join(results))

<svg xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" style="fill-opacity:1; color-rendering:auto; color-interpolation:auto; text-rendering:auto; stroke:black; stroke-linecap:square; stroke-miterlimit:10; shape-rendering:auto; stroke-opacity:1; fill:black; stroke-dasharray:none; font-weight:normal; stroke-width:1; font-family:'Dialog'; font-style:normal; stroke-linejoin:miter; font-size:12px; stroke-dashoffset:0; image-rendering:auto;" width="900" height="900"><defs id="genericDefs"/><g><g style="stroke-linecap:round; fill:white; stroke:white;"><rect x="0" width="900" height="900" y="0" style="stroke:none;"/></g><g style="stroke-linecap:round;" transform="translate(431.0456,173.9697) rotate(350.1284)"><ellipse rx="74.4221" ry="48.0768" style="fill:none;" cx="0" cy="0"/><ellipse rx="70.2875" ry="45.4058" style="fill:none;" cx="0" cy="0"/><ellipse rx="66.153" ry="42.7349" style="fill:none;" cx="0" cy="0"/><ellipse rx="62.0184" ry="40.064" style="fill:none;" c

In [19]:
display(SVG(data="".join(results)))

ExpatError: duplicate attribute: line 1, column 4105